# Step 7 — Il knowledge graph, esplorato

Il documento `docs/07_knowledge_graph.md` spiega **perché** il grafo ha questa
forma. Questo notebook lo apre e ci guarda dentro: quanto è grande, che forma
hanno le sue componenti, e soprattutto **che cosa succederebbe alla soglia di
consenso che lo step 8 dovrà scegliere**.

> **Nessun output di questo notebook contiene testo clinico.** Le menzioni nel
> grafo portano gli offset, non le citazioni; i conteggi sono aggregati. È la
> stessa regola dei notebook 01 e 02, e `src/privacy.py` la verifica
> meccanicamente su tutto il repository.

In [ ]:
import sys
from collections import Counter
from pathlib import Path

RADICE = Path.cwd().parent
sys.path.insert(0, str(RADICE / 'src'))

from grafo import costruisci, CT, PIPELINE, ATC, ICD, PROV
from rdflib import Literal
from rdflib.namespace import RDF, SKOS

## 1. Costruire il grafo

Si ricostruisce invece di rileggere il Turtle: il parser di rdflib è puro
Python e su 1,2 milioni di triple impiega minuti, mentre la costruzione dalle
uscite delle pipeline ne impiega **dieci secondi**. È una misura, non
un'impressione, ed è il motivo per cui `interroga.py` ricostruisce di default.

In [ ]:
%%time
costruzione = costruisci(
    cartella_b=RADICE / 'data' / 'processed' / 'pipeline_b_deepseek_1000')
g = costruzione.grafo
costruzione.conteggi

## 2. Quante pipeline sostengono ciascuna asserzione

È la distribuzione da cui dipende ogni soglia di consenso dello step 8.

In [ ]:
consenso = Counter(
    int(n) for n in g.objects(None, CT.numeroPipeline))
totale = sum(consenso.values())
for k in sorted(consenso):
    print(f'{k} pipeline: {consenso[k]:6}  ({consenso[k]/totale:5.1%})')

### Che cosa costerebbe ciascuna soglia

La domanda non è «quante asserzioni restano» ma **quante se ne perdono, e di
chi erano**. Lo step 6bis ha misurato che la pipeline B ha un richiamo del 70%
sulle condizioni contro il 20% delle altre due: se le asserzioni scartate da
una soglia fossero in maggioranza sue, la soglia starebbe buttando via proprio
il contributo della sola pipeline che vede la maggior parte delle condizioni.

In [ ]:
for soglia in (1, 2, 3):
    tenute = sum(v for k, v in consenso.items() if k >= soglia)
    print(f'soglia >= {soglia}: {tenute:6} tenute  '
          f'({tenute/totale:5.1%}), {totale-tenute:6} scartate')

## 3. Chi contribuisce da solo

Per ogni asserzione vista da **una sola** pipeline, quale.

In [ ]:
solitarie = Counter()
for a in g.subjects(CT.numeroPipeline, Literal(1)):
    for m in g.objects(a, PROV.wasDerivedFrom):
        for p in g.objects(m, PROV.wasAttributedTo):
            solitarie[str(p).rsplit('/', 1)[-1]] += 1
for sigla, n in solitarie.most_common():
    print(f'{sigla}: {n:6}')

**È lo step 6bis confermato sul corpus intero con un metodo diverso.** Il
riferimento annotato misurava il richiamo su 25 referti letti a mano; qui, su
1 000 referti e senza alcuna annotazione, si vede la stessa cosa da un altro
lato — quanta informazione esiste *solo* grazie a ciascuna pipeline.

Due misure indipendenti, la stessa conclusione: **una soglia di consenso a due
pipeline non scarterebbe «le asserzioni dubbie», scarterebbe quasi per intero
il contributo esclusivo della pipeline con il richiamo più alto.** È il
compromesso centrale dello step 8, e adesso è un numero invece che un'opinione.

## 4. Stato e soggetto: ciò che un sistema ingenuo sbaglierebbe

Le condizioni negate, incerte e dei familiari sono quelle che un sistema che
leggesse il grafo come «elenco delle malattie del paziente» attribuirebbe per
errore. Qui si contano.

In [ ]:
assi = Counter()
for m in g.subjects(CT.tipoEntita, Literal('condizione')):
    stato = g.value(m, CT.stato)
    soggetto = g.value(m, CT.soggetto)
    if stato is not None:
        assi[(str(stato), str(soggetto))] += 1
tot = sum(assi.values())
for (stato, soggetto), n in assi.most_common():
    print(f'{stato:10} {soggetto:10} {n:6}  ({n/tot:5.1%})')
sbagliate = tot - assi[('affermato', 'paziente')]
print(f'\nnon attuali del paziente: {sbagliate} ({sbagliate/tot:.1%})')

I due assi sono **indipendenti** di proposito: «familiarità negativa per
cardiopatia ischemica» è insieme familiare e negata, e comprimerli in un solo
campo perderebbe l'una o l'altra informazione. Le righe `negato` + `familiare`
lo dimostrano: esistono, e non sono poche.

## 5. La gerarchia ATC, che serve allo step 11

`skos:broader` risale dal principio attivo al livello anatomico. La metrica
gerarchica dello step 11 misurerà *quanto* due codici siano vicini: due terapie
diverse nello stesso gruppo terapeutico non sono un errore quanto due terapie
in gruppi diversi. Qui si verifica che la scala esista davvero.

In [ ]:
acido = ATC['B01AC06']  # acido acetilsalicilico, il farmaco piu' comune nel corpus
print('catena gerarchica:')
nodo = acido
while True:
    etichetta = g.value(nodo, SKOS.prefLabel)
    livello = g.value(nodo, CT.livelloATC)
    codice = g.value(nodo, SKOS.notation)
    print(f'  {str(codice):8} {str(livello):14} {etichetta}')
    padri = list(g.objects(nodo, SKOS.broader))
    if not padri:
        break
    nodo = padri[0]

In [ ]:
# Quanti codici a ciascun livello, e quanti di essi compaiono davvero nei referti
usati = {str(c) for c in g.objects(None, CT.risolveA)}
per_livello = Counter()
usati_per_livello = Counter()
for c in g.subjects(RDF.type, SKOS.Concept):
    livello = g.value(c, CT.livelloATC)
    if livello is None:
        continue
    per_livello[str(livello)] += 1
    if str(c) in usati:
        usati_per_livello[str(livello)] += 1
for livello in ('anatomico', 'terapeutico', 'farmacologico', 'chimico', 'sostanza'):
    print(f'{livello:14} {per_livello[livello]:5} codici, '
          f'{usati_per_livello[livello]:5} usati nei referti')

Il grafo contiene la gerarchia **intera**, non solo i codici che compaiono nei
referti. Non è spreco: senza i livelli superiori non ci sarebbe nulla rispetto
a cui misurare una distanza, e la metrica dello step 11 non avrebbe un albero
su cui muoversi.

## 6. Quanto resta senza codice

Il vincolo del progetto è **conservare, non cancellare**: le menzioni che
nessun risolutore ha saputo codificare restano nel grafo, marcate `ct:irrisolta`.
Questo conteggio dice quanto lavoro di normalizzazione resta scoperto — ed è
informazione che una pipeline che scartasse gli irrisolti renderebbe invisibile.

In [ ]:
irrisolte = Counter()
totali = Counter()
for m in g.subjects(RDF.type, CT.Menzione):
    tipo = str(g.value(m, CT.tipoEntita))
    totali[tipo] += 1
    if g.value(m, CT.irrisolta) is not None:
        irrisolte[tipo] += 1
for tipo in totali:
    print(f'{tipo:12} {irrisolte[tipo]:6} irrisolte su {totali[tipo]:6} '
          f'({irrisolte[tipo]/totali[tipo]:5.1%})')

## 7. La verifica che il testo clinico non sia entrato

Il grafo viene serializzato in un file che può circolare. Le menzioni portano
**gli offset, non il testo**. Qui lo si verifica sui dati veri invece di
fidarsi del test unitario, che lavora su una menzione finta.

In [ ]:
from data_loading import carica_dataset

record, _ = carica_dataset(RADICE / 'data' / 'raw' / 'anamnesiterapie.txt')
# Un campione di frasi vere, lunghe abbastanza da essere identificanti.
campione = [(r.testo_anamnesi or '')[40:120] for r in record[:200]]
campione = [s for s in campione if len(s) >= 60]

turtle = g.serialize(format='turtle')
trovate = [s for s in campione if s in turtle]
print(f'frasi di referto cercate nel Turtle: {len(campione)}')
print(f'trovate: {len(trovate)}')
assert not trovate, 'testo clinico nel grafo serializzato'

---

## Che cosa lascia decidere allo step 8

Il grafo **non decide**: non fonde le contraddizioni, non sceglie fra un
`affermato` e un `negato` quando due pipeline dissentono, non scarta gli
irrisolti. Conserva tutto e segna chi dice cosa.

Quello che questo notebook mostra è che le domande dello step 8 hanno adesso
una risposta numerica invece che un'intuizione:

* **una soglia di consenso costa**, e il costo non è distribuito a caso: cade
  quasi per intero sulla pipeline con il richiamo più alto;
* **poco più di una menzione di condizione su dieci non è una condizione
  attuale del paziente**, ed è negata, incerta o di un familiare;
* **i codici che vengono dal solo gazetteer sono isolabili**, e lo step 6 ha
  mostrato che sono la categoria di errore più insidiosa, perché sbagliata e
  già codificata.

Il filtro dello step 8 sarà simbolico e ispezionabile — mai un LLM, mai il
dataset — e queste sono le tre leve che avrà.